# ⚡ Starfelt — Training Efficiency Layer

**Stop losing progress when Colab disconnects. Track cost live. One import.**

- Auto-checkpoints to Google Drive — resume after any disconnect
- Live cost + loss table updates in this cell as you train
- Works with any PyTorch model, zero config

[![GitHub](https://img.shields.io/badge/GitHub-victorachede/starfelt-black)](https://github.com/victorachede/starfelt)

In [ ]:
# Step 1 — Install
!pip install -q git+https://github.com/victorachede/starfelt.git

In [ ]:
# Step 2 — Mount Drive so checkpoints survive disconnects
from starfelt.notebook import mount_drive
mount_drive()

In [ ]:
# Step 3 — Train with live display
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from starfelt import Trainer
from starfelt.notebook import StarfeltDisplay

# Toy model — replace with your real model
model = nn.Sequential(
    nn.Linear(16, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Toy dataset — replace with yours
x = torch.randn(256, 16)
y = torch.randn(256, 1)
train_loader = DataLoader(TensorDataset(x, y), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(x[:64], y[:64]), batch_size=32)

# Live display — updates this cell in place
display = StarfeltDisplay()

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    loss_fn=nn.MSELoss(),
    val_loader=val_loader,
    epochs=5,
    early_stop_patience=3,
    checkpoint_every_epochs=1,
    notebook_display=display,
)

result = trainer.fit()

In [ ]:
# Step 4 — Inspect the run
print(f"Run ID:          {result.run_id}")
print(f"Epochs:          {result.epochs_completed}")
print(f"Final loss:      {result.final_loss:.4f}")
print(f"Final val loss:  {result.final_val_loss}")
print(f"Tracked cost:    ${result.cost_usd:.4f}")
print(f"Checkpoint:      {result.checkpoint_path}")
print(f"Stopped early:   {result.stopped_early}")

In [ ]:
# Step 5 — If Colab disconnected mid-run, resume from Drive checkpoint
# Just re-run this cell after reconnecting

import os
run_id = result.run_id  # or paste your run_id string here
ckpt = f"/content/drive/MyDrive/.starfelt/checkpoints/{run_id}/latest.pt"

if os.path.exists(ckpt):
    os.environ["STARFELT_RESUME_FROM"] = ckpt
    os.environ["STARFELT_RUN_ID"] = run_id
    print(f"Resuming from: {ckpt}")

    resume_display = StarfeltDisplay()
    resume_trainer = Trainer(
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        loss_fn=nn.MSELoss(),
        val_loader=val_loader,
        epochs=5,
        notebook_display=resume_display,
    )
    resumed = resume_trainer.fit()
    print(f"Resumed — total epochs: {resumed.epochs_completed}")
else:
    print("No checkpoint found — run Step 3 first.")

## What just happened?

- Starfelt auto-detected Colab and saved checkpoints to Google Drive
- If this tab had disconnected mid-training, Step 5 would have resumed exactly where it stopped
- The live table showed loss, val loss, learning rate, and cost per epoch

**Next steps:**
- Replace the toy model with your real model
- Add `wandb=True` to Trainer for experiment tracking
- Star the repo ⭐ → [github.com/victorachede/starfelt](https://github.com/victorachede/starfelt)